# 01 — Data Exploration: 3W Dataset

This notebook demonstrates how to load and explore the **Petrobras 3W dataset** using the `ThreeWLoader` class from `qml.samarone_junior`.  
The 3W dataset contains multivariate time-series recordings from offshore oil wells, labelled with 9 fault event types plus a normal operating class.

## 1. Setup and Imports

We import `ThreeWLoader` from the project's `loaders` subpackage. The loader discovers Parquet files organised by class label in the 3W directory structure.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from qml.samarone_junior.loaders import ThreeWLoader

# Point loader at the local 3W dataset directory
loader = ThreeWLoader(data_path="../../data/samarone_junior/3w")
print(f"Data path: {loader.data_path}")

## 2. Event Class Catalogue

The `EVENT_NAMES` dictionary maps integer class labels (0–9) to human-readable event descriptions.  
Class 0 is normal operation; classes 1–9 are anomalous events.

In [ ]:
# Display the event catalogue
for cls_id, name in ThreeWLoader.EVENT_NAMES.items():
    print(f"  Class {cls_id}: {name}")

print(f"\nSensor columns: {ThreeWLoader.SENSORS}")

## 3. Discover Available Instances

For each event class, the loader scans the dataset directory and returns a list of Parquet file paths. We count how many instances exist per class and source type (real, simulated, drawn).

In [ ]:
# Count instances per class
class_counts = {}
for cls_id in ThreeWLoader.EVENT_NAMES:
    files = loader.list_instances(cls_id)
    class_counts[cls_id] = len(files)

counts_df = pd.DataFrame([
    {"class": k, "event": ThreeWLoader.EVENT_NAMES[k], "n_instances": v}
    for k, v in class_counts.items()
])
counts_df

## 4. Load and Visualise a Sample Time Series

We load a single Parquet instance from class 1 (Abrupt BSW Increase) and plot the 5 sensor channels over time.

In [ ]:
# Load one instance from class 1
instances_cls1 = loader.list_instances(1)
if instances_cls1:
    sample_path = instances_cls1[0]
    df = pd.read_parquet(sample_path)
    print(f"Loaded: {sample_path}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    display(df.head())
else:
    print("No instances found for class 1 — check data_path.")

In [ ]:
# Plot sensor channels
if instances_cls1:
    fig, axes = plt.subplots(len(ThreeWLoader.SENSORS), 1, figsize=(12, 10), sharex=True)
    for i, sensor in enumerate(ThreeWLoader.SENSORS):
        if sensor in df.columns:
            axes[i].plot(df[sensor].values, linewidth=0.5)
            axes[i].set_ylabel(sensor, fontsize=9)
            axes[i].tick_params(labelsize=8)
    axes[-1].set_xlabel("Timestep")
    fig.suptitle("Class 1 — Abrupt BSW Increase (sample)", fontsize=12)
    plt.tight_layout()
    plt.show()

## 5. Summary

- The 3W dataset has **10 classes** (0 = Normal, 1–9 = anomalous events).
- Each instance is a Parquet file with 5 sensor columns sampled at 1 Hz.
- `ThreeWLoader` provides a clean API to discover and load instances by class and source type.